# PopOut – Adversarial Search with MCTS
**Artificial Intelligence 2025/2026** · Grupo 3

---

## 1. Introduction

**PopOut** is a variant of Connect-4 in which, in addition to the standard _drop_ move (place a disc at the top of a column), a player may also _pop_ one of their own discs out from the **bottom row** of any column — every disc above it then falls one row down.

### Rules summary

| Rule | Description |
|------|-------------|
| Drop | Place a disc at the top of a column (gravity drops it to lowest empty cell) |
| Pop  | Remove YOUR OWN disc from the bottom row of a column; all discs above shift down |
| Win  | First player to connect 4 discs (horizontal / vertical / diagonal) wins |
| Special 1 | If a _pop_ creates 4-in-a-row for **both** players simultaneously, the player who popped **wins** |
| Special 2 | If the board is **full** and no drop is possible, the player to move may pop (if able) or declare a **draw** |
| Special 3 | If the same board state is repeated **3 times**, either player may declare a **draw** |

### Implementation overview

| File | Contents |
|------|----------|
| `popout_game.py` | `PopOutGame` class – all game logic, move application, win detection |
| `mcts.py` | `MCTSNode`, `MCTS` and factory helpers for the different variants |
| `main.py` | Text-based interactive runner (Human vs Human / Human vs AI / AI vs AI) |
| `PopOut_IA.ipynb` | **This notebook** – documentation, experiments, dataset generation |

---
## 2. PopOut Game Implementation

In [ ]:
# All game logic lives in popout_game.py
# This cell re-defines the class inline so the notebook is self-contained.

import math, random, time, csv, os
from copy import deepcopy

EMPTY   = 0
PLAYER1 = 1   # 'X'
PLAYER2 = 2   # 'O'

class PopOutGame:
    """
    Immutable-style game state for PopOut.
    All moves return a NEW state; the original is never modified.
    """
    EMPTY = EMPTY; PLAYER1 = PLAYER1; PLAYER2 = PLAYER2

    def __init__(self, rows=6, cols=7):
        self.rows = rows
        self.cols = cols
        self.board = [[EMPTY]*cols for _ in range(rows)]
        self.current_player = PLAYER1
        self.state_history = {}
        self.winner = None
        self.game_over = False
        self._record_state()

    # ── State tracking ──────────────────────────────────────────────────────
    def _get_state_key(self):
        return (tuple(tuple(r) for r in self.board), self.current_player)
    def _record_state(self):
        k = self._get_state_key()
        self.state_history[k] = self.state_history.get(k, 0) + 1
    def get_state_repetitions(self):
        return self.state_history.get(self._get_state_key(), 0)
    def is_repetition_draw_available(self):
        return self.get_state_repetitions() >= 3

    # ── Board queries ────────────────────────────────────────────────────────
    def is_board_full(self):
        return all(self.board[0][c] != EMPTY for c in range(self.cols))
    def get_drop_moves(self):
        return [c for c in range(self.cols) if self.board[0][c] == EMPTY]
    def get_pop_moves(self):
        return [c for c in range(self.cols) if self.board[self.rows-1][c] == self.current_player]
    def get_all_moves(self):
        if self.game_over: return []
        return [('drop',c) for c in self.get_drop_moves()] + [('pop',c) for c in self.get_pop_moves()]

    # ── Move application ─────────────────────────────────────────────────────
    def apply_move(self, move_type, col):
        if self.game_over: return None
        g = self._copy()
        if move_type == 'drop':
            if g.board[0][col] != EMPTY: return None
            row = self.rows-1
            while row >= 0 and g.board[row][col] != EMPTY: row -= 1
            if row < 0: return None
            g.board[row][col] = self.current_player
            g._resolve_after_drop()
        elif move_type == 'pop':
            if g.board[self.rows-1][col] != self.current_player: return None
            for r in range(self.rows-1, 0, -1):
                g.board[r][col] = g.board[r-1][col]
            g.board[0][col] = EMPTY
            g._resolve_after_pop()
        else: return None
        return g

    def _resolve_after_drop(self):
        p1, p2 = self._check_four(PLAYER1), self._check_four(PLAYER2)
        if p1 or p2:
            self.winner = self.current_player if (p1 and p2) else (PLAYER1 if p1 else PLAYER2)
            self.game_over = True
        else:
            self._switch()

    def _resolve_after_pop(self):
        p1, p2 = self._check_four(PLAYER1), self._check_four(PLAYER2)
        if p1 or p2:
            self.winner = self.current_player  # pop player always wins on simultaneous
            self.game_over = True
        else:
            self._switch()

    def _switch(self):
        self.current_player = PLAYER2 if self.current_player == PLAYER1 else PLAYER1
        self._record_state()

    def _check_four(self, p):
        b, R, C = self.board, self.rows, self.cols
        for r in range(R):
            for c in range(C-3):
                if b[r][c]==b[r][c+1]==b[r][c+2]==b[r][c+3]==p: return True
        for r in range(R-3):
            for c in range(C):
                if b[r][c]==b[r+1][c]==b[r+2][c]==b[r+3][c]==p: return True
        for r in range(R-3):
            for c in range(C-3):
                if b[r][c]==b[r+1][c+1]==b[r+2][c+2]==b[r+3][c+3]==p: return True
        for r in range(R-3):
            for c in range(3,C):
                if b[r][c]==b[r+1][c-1]==b[r+2][c-2]==b[r+3][c-3]==p: return True
        return False

    def _copy(self):
        g = object.__new__(PopOutGame)
        g.rows=self.rows; g.cols=self.cols
        g.board=[r[:] for r in self.board]
        g.current_player=self.current_player
        g.state_history=dict(self.state_history)
        g.winner=self.winner; g.game_over=self.game_over
        return g

    # ── Display ──────────────────────────────────────────────────────────────
    @staticmethod
    def player_symbol(p): return 'X' if p==PLAYER1 else 'O'
    def display(self):
        s = {EMPTY:'-', PLAYER1:'X', PLAYER2:'O'}
        print()
        for row in self.board: print(''.join(s[c] for c in row))
        print('1234567'[:self.cols])
        if not self.game_over:
            print(f"\nIt is {self.player_symbol(self.current_player)}'s turn.")
        elif self.winner: print(f"\n{self.player_symbol(self.winner)} wins!")
        else: print("\nDraw!")
        print()
    def get_board_flat(self): return [c for row in self.board for c in row]

print('PopOutGame defined.')

### 2.1 Quick demo – a few manual moves

In [ ]:
g = PopOutGame()
# Drop some discs
for col in [3, 3, 3, 4, 2, 5]:
    g = g.apply_move('drop', col)
g.display()
print('Available moves:', g.get_all_moves())

In [ ]:
# Demo of pop move: X has disc at bottom of col 3 (index 3)
# Drop enough to fill col 3 so X reaches the bottom
g2 = PopOutGame()
for col in [3,3,3,3,3,3]:  # fill column 3
    g2 = g2.apply_move('drop', col)
    if g2 is None:
        print('Column full'); break
if g2:
    g2.display()
    print('Pop moves available for X:', g2.get_pop_moves())
    g3 = g2.apply_move('pop', 3)
    if g3:
        print('After popping column 3:')
        g3.display()

---
## 3. Monte Carlo Tree Search (MCTS)

MCTS is an adversarial search algorithm that uses **random simulation (Monte Carlo)** to estimate the value of each move without needing a hand-crafted evaluation function.

### 3.1 The Four Phases

```
┌─────────────────────────────────────────────────────────────┐
│  SELECTION   │ EXPANSION │  SIMULATION   │ BACKPROPAGATION │
│  (UCT)       │ (1 child) │  (rollout)    │  (update stats) │
└─────────────────────────────────────────────────────────────┘
```

1. **Selection** – start at the root and traverse the existing tree by always choosing the child that maximises the **UCT** value, until a node that is not fully expanded (or a terminal node) is reached.
2. **Expansion** – add one new child node for a randomly chosen untried move.
3. **Simulation (Rollout)** – from the new node, play randomly (or with a heuristic) until the game ends.
4. **Backpropagation** – propagate the result back up to the root, updating visit counts and win sums.

### 3.2 UCT Formula

$$\text{UCT}(v') = \underbrace{\frac{Q(v')}{N(v')}}_{\text{exploitation}} + C \cdot \underbrace{\sqrt{\frac{\ln N(v)}{N(v')}}}_{\text{exploration}}$$

where:
- $Q(v')$ = cumulative wins at child $v'$ (from **child's** player perspective)
- $N(v')$ = visit count of child $v'$
- $N(v)$ = visit count of parent $v$
- $C$ = exploration constant (typically $\sqrt{2} \approx 1.414$)

Because $Q/N$ is stored from the **child's** player perspective (the opponent of the current node's player), the parent selects using $(1 - Q/N) + C\sqrt{\ldots}$ — which inverts the perspective back to the current player.

### 3.3 Variants Implemented

| Variant | Key difference |
|---------|----------------|
| Standard UCT | $C = \sqrt{2}$ |
| High exploration | $C = 2.5$ — broader tree, shallower search |
| Low exploration (greedy) | $C = 0.5$ — exploits more, explores less |
| Progressive widening | cap on children per node (`max_children`) |
| Heuristic rollout | one-step look-ahead: prefer immediate wins / blocks |
| UCT-Tuned | empirical-variance bound replaces plain $\sqrt{\ln N / n}$ |

In [ ]:
# ─── MCTS Node ───────────────────────────────────────────────────────────────

class MCTSNode:
    __slots__ = ('game_state','parent','move','children','wins','visits','untried_moves')

    def __init__(self, game_state, parent=None, move=None):
        self.game_state    = game_state
        self.parent        = parent
        self.move          = move
        self.children      = []
        self.wins          = 0.0
        self.visits        = 0
        self.untried_moves = list(game_state.get_all_moves())

    def is_terminal(self):       return self.game_state.game_over
    def is_fully_expanded(self): return len(self.untried_moves) == 0

    def uct_value(self, C):
        """UCT from parent's perspective: (1 - Q/N) + C*sqrt(ln(parent_N)/N)"""
        if self.visits == 0: return float('inf')
        return (1.0 - self.wins/self.visits) + C * math.sqrt(math.log(self.parent.visits)/self.visits)

    def uct_tuned_value(self, C):
        """UCT-Tuned with empirical variance bound."""
        if self.visits == 0: return float('inf')
        q = self.wins / self.visits
        var = q - q*q + math.sqrt(2*math.log(self.parent.visits)/self.visits)
        return (1.0-q) + C*math.sqrt(math.log(self.parent.visits)/self.visits * min(0.25,var))

    def best_child(self, C, tuned=False):
        fn = self.uct_tuned_value if tuned else self.uct_value
        return max(self.children, key=lambda ch: fn.__func__(ch, C) if False else ch.uct_tuned_value(C) if tuned else ch.uct_value(C))

# ─── MCTS ────────────────────────────────────────────────────────────────────

class MCTS:
    def __init__(self, iterations=1000, C=math.sqrt(2),
                 max_children=None, rollout='random', tuned=False, name='MCTS'):
        self.iterations   = iterations
        self.C            = C
        self.max_children = max_children
        self.rollout      = rollout
        self.tuned        = tuned
        self.name         = name

    def get_best_move(self, state):
        root = MCTSNode(state)
        for _ in range(self.iterations):
            leaf = self._select(root)
            if not leaf.is_terminal():
                leaf = self._expand(leaf)
            result = self._simulate(leaf)
            self._backpropagate(leaf, result)
        if not root.children:
            moves = state.get_all_moves()
            return random.choice(moves) if moves else None
        return max(root.children, key=lambda ch: ch.visits).move

    def get_move_stats(self, state):
        root = MCTSNode(state)
        for _ in range(self.iterations):
            leaf = self._select(root)
            if not leaf.is_terminal(): leaf = self._expand(leaf)
            self._backpropagate(leaf, self._simulate(leaf))
        return {ch.move: {'visits':ch.visits,
                          'win_rate': ch.wins/ch.visits if ch.visits else 0.0}
                for ch in root.children}

    def _select(self, node):
        while not node.is_terminal() and node.is_fully_expanded():
            node = node.best_child(self.C, self.tuned)
        return node

    def _expand(self, node):
        if not node.untried_moves: return node
        if self.max_children and len(node.children) >= self.max_children:
            return node.best_child(self.C, self.tuned) if node.children else node
        move = random.choice(node.untried_moves)
        ns   = node.game_state.apply_move(*move)
        if ns is None:
            node.untried_moves.remove(move)
            return self._expand(node)
        child = MCTSNode(ns, parent=node, move=move)
        node.untried_moves.remove(move)
        node.children.append(child)
        return child

    def _simulate(self, node):
        state  = node.game_state._copy()
        player = node.game_state.current_player
        for _ in range(200):
            if state.game_over: break
            moves = state.get_all_moves()
            if not moves: break
            move = self._heuristic_select(state, moves) if self.rollout=='heuristic' else random.choice(moves)
            ns   = state.apply_move(*move)
            if ns: state = ns
        if state.winner == player: return 1.0
        if state.winner is None:   return 0.5
        return 0.0

    def _heuristic_select(self, state, moves):
        p = state.current_player
        opp = PLAYER2 if p==PLAYER1 else PLAYER1
        for m in moves:
            s = state.apply_move(*m)
            if s and s.winner == p: return m
        for m in moves:
            s = state.apply_move(*m)
            if s and s.winner == opp: return m
        return random.choice(moves)

    def _backpropagate(self, node, result):
        while node:
            node.visits += 1
            node.wins   += result
            result = 1.0 - result
            node = node.parent

print('MCTS defined.')

### 3.4 Sanity check – AI picks the winning move

In [ ]:
# Set up a state where X (PLAYER1) can win immediately by dropping in col 4
g = PopOutGame()
for c in [0,1,1,2,2,3]:   # X has 3 in a row at bottom; dropping col 3 (idx) wins
    g = g.apply_move('drop', c)
g.display()

ai = MCTS(iterations=500, C=math.sqrt(2), name='Sanity-Check')
move = ai.get_best_move(g)
print(f'AI chose: {move[0].upper()} column {move[1]+1}')
g2 = g.apply_move(*move)
g2.display()

---
## 4. Game Modes

The three required modes are implemented below. For automated (CvC) games the `auto_play` helper is used; for interactive play see `main.py`.

In [ ]:
def auto_play(ai1, ai2, verbose=False):
    """
    Run one complete game between two MCTS agents.
    Returns (winner, n_moves).  winner is PLAYER1, PLAYER2, or None (draw).
    """
    game = PopOutGame()
    n = 0
    while not game.game_over:
        if game.is_repetition_draw_available():
            if verbose: print('[Draw by repetition]')
            return None, n
        ai = ai1 if game.current_player == PLAYER1 else ai2
        move = ai.get_best_move(game)
        if move is None: break
        ng = game.apply_move(*move)
        if ng is None: break
        game = ng
        n += 1
    if verbose: game.display()
    return game.winner, n

# Quick demo: one CvC game with verbose output
ai_x = MCTS(iterations=300, C=math.sqrt(2), name='X-Standard')
ai_o = MCTS(iterations=300, C=math.sqrt(2), name='O-Standard')

winner, moves = auto_play(ai_x, ai_o, verbose=True)
sym = PopOutGame.player_symbol(winner) if winner else 'Draw'
print(f'Result: {sym}  |  Moves played: {moves}')

---
## 5. Experiments – Comparing MCTS Variants

We compare six configurations in a round-robin tournament.
Each match-up is played **N_GAMES** times, alternating who plays X.

In [ ]:
# Define the variants
ITERS = 400   # keep low for notebook speed; increase for better statistics

variants = [
    MCTS(iterations=ITERS, C=math.sqrt(2),  name='Standard (C=√2)'),
    MCTS(iterations=ITERS, C=2.5,           name='High-Expl (C=2.5)'),
    MCTS(iterations=ITERS, C=0.5,           name='Low-Expl  (C=0.5)'),
    MCTS(iterations=ITERS, C=math.sqrt(2),  max_children=4, name='Prog-Widening(k=4)'),
    MCTS(iterations=ITERS, C=math.sqrt(2),  rollout='heuristic', name='Heuristic-Rollout'),
    MCTS(iterations=ITERS, C=math.sqrt(2),  tuned=True, name='UCT-Tuned'),
]
print('Variants:')
for v in variants: print(f'  {v.name}')

In [ ]:
def tournament(variants, n_games=10):
    """
    Round-robin tournament: each pair plays n_games games (half as X, half as O).
    Returns a win-matrix W[i][j] = wins for variant i against variant j.
    """
    n = len(variants)
    W = [[0]*n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i == j: continue
            for game_idx in range(n_games):
                if game_idx % 2 == 0:
                    ai1, ai2 = variants[i], variants[j]
                    p_i = PLAYER1
                else:
                    ai1, ai2 = variants[j], variants[i]
                    p_i = PLAYER2
                winner, _ = auto_play(ai1, ai2)
                if winner == p_i:
                    W[i][j] += 1
    return W

N_GAMES = 6   # increase for more reliable results
print(f'Running tournament ({len(variants)} variants × {N_GAMES} games each pair)…')
t0 = time.time()
W = tournament(variants, n_games=N_GAMES)
print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
# Display results table
n = len(variants)
names = [v.name for v in variants]
totals = [sum(W[i]) for i in range(n)]

col_w = max(len(nm) for nm in names) + 2
header = f"{'Variant':<{col_w}}" + ''.join(f'{i:>4}' for i in range(n)) + '  TOTAL'
print(header)
print('-' * len(header))
for i, nm in enumerate(names):
    row = f'{nm:<{col_w}}' + ''.join(f'{W[i][j]:>4}' if j!=i else '   -' for j in range(n))
    row += f'  {totals[i]:>4}'
    print(row)
print()
best_idx = totals.index(max(totals))
print(f'Best variant: {names[best_idx]}  ({totals[best_idx]} wins)')

### 5.1 Effect of iteration budget

In [ ]:
# Compare Standard MCTS at different budgets vs a weak baseline (50 iterations)
baseline = MCTS(iterations=50, C=math.sqrt(2), name='Weak(50)')
budgets  = [100, 200, 400, 800]
REPS     = 8

print(f'Win rate of Standard MCTS(n) vs Weak(50) over {REPS} games each:')
for n_iter in budgets:
    strong = MCTS(iterations=n_iter, C=math.sqrt(2), name=f'Strong({n_iter})')
    wins = 0
    for k in range(REPS):
        ai1, ai2 = (strong, baseline) if k%2==0 else (baseline, strong)
        p_strong = PLAYER1 if k%2==0 else PLAYER2
        w, _ = auto_play(ai1, ai2)
        if w == p_strong: wins += 1
    print(f'  n={n_iter:>4}: {wins}/{REPS} wins  ({100*wins/REPS:.0f}%)')

### 5.2 UCT move statistics visualisation

In [ ]:
# Show the visit/win-rate distribution for each possible move from the start state
start = PopOutGame()
ai = MCTS(iterations=1000, C=math.sqrt(2), name='Stats')
stats = ai.get_move_stats(start)

print('Move statistics from the initial position (X to move):')
print(f'{"Move":<12} {"Visits":>8} {"Win-rate":>10}')
print('-' * 32)
for move, s in sorted(stats.items(), key=lambda x: -x[1]['visits']):
    mtype, col = move
    print(f'{mtype.upper()+" col "+str(col+1):<12} {s["visits"]:>8} {s["win_rate"]:>10.3f}')

---
## 6. Dataset Generation for Decision Trees

We generate a dataset of `(state, best_move)` pairs by running MCTS on many game positions.

**State encoding** (42 features for a 6×7 board):
- Each cell encoded as: `0` = empty, `1` = current player, `2` = opponent.
- This normalises the representation regardless of which player is to move.

**Label**: `move_type` (`drop`/`pop`) concatenated with the column index (0–6).
Example: `drop_3`, `pop_1`.

In [ ]:
def encode_state(game):
    """
    Encode the board from the CURRENT PLAYER's perspective.
    own disc = 1, opponent = 2, empty = 0.
    Returns a flat list of 42 integers.
    """
    p = game.current_player
    opp = PLAYER2 if p == PLAYER1 else PLAYER1
    enc = []
    for row in game.board:
        for cell in row:
            if cell == p:   enc.append(1)
            elif cell == opp: enc.append(2)
            else:           enc.append(0)
    return enc

def generate_dataset(n_games=50, mcts_iterations=300, output_csv='popout_dataset.csv'):
    """
    Play `n_games` self-play games using MCTS and record (state, move) pairs.
    Each state is encoded as a 42-dim vector; each move is a string label.
    """
    ai = MCTS(iterations=mcts_iterations, C=math.sqrt(2), rollout='heuristic', name='DataGen')
    rows = []
    feature_names = [f'cell_{r}_{c}' for r in range(6) for c in range(7)]

    for game_idx in range(n_games):
        game = PopOutGame()
        while not game.game_over:
            if game.is_repetition_draw_available():
                break
            # Query MCTS for the best move
            move = ai.get_best_move(game)
            if move is None: break
            # Record (state, move)
            state_enc = encode_state(game)
            move_label = f'{move[0]}_{move[1]}'
            rows.append(state_enc + [move_label])
            # Apply the move
            ng = game.apply_move(*move)
            if ng is None: break
            game = ng
        if (game_idx + 1) % 10 == 0:
            print(f'  Generated {game_idx+1}/{n_games} games  ({len(rows)} samples so far)')

    # Write CSV
    header = feature_names + ['move']
    with open(output_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(header)
        writer.writerows(rows)

    print(f'\nDataset saved to {output_csv}  ({len(rows)} samples)')
    return rows, header

print('Generating dataset (50 games)…')
t0 = time.time()
dataset_rows, header = generate_dataset(n_games=50, mcts_iterations=300)
print(f'Done in {time.time()-t0:.1f}s')

In [ ]:
# Quick inspection of the dataset
labels = [r[-1] for r in dataset_rows]
label_counts = {}
for l in labels: label_counts[l] = label_counts.get(l, 0) + 1

print(f'Total samples : {len(dataset_rows)}')
print(f'Unique moves  : {len(label_counts)}')
print('\nMove distribution (top 15):')
for move, cnt in sorted(label_counts.items(), key=lambda x: -x[1])[:15]:
    bar = '#' * (cnt // 5)
    print(f'  {move:<12} {cnt:>5}  {bar}')

---
## 7. Summary and Discussion

### What was implemented

1. **PopOut game** (`PopOutGame`) – complete rule set including pop moves, all three special rules (simultaneous 4-in-a-row, full-board draw, repetition draw).

2. **MCTS with UCT** – standard four-phase algorithm with the UCT selection formula. Reward signal: 1 win / 0.5 draw / 0 loss from the current node's player perspective; backpropagation alternates with `1 − r`.

3. **Six variants explored**:
   - Different exploration constants ($C \in \{0.5, \sqrt{2}, 2.5\}$)
   - Progressive widening (cap on children per node)
   - Heuristic rollout (1-step look-ahead for immediate wins/blocks)
   - UCT-Tuned (variance-based exploration bound)

4. **Three game modes** – human vs human, human vs computer, computer vs computer (see `main.py` for the interactive runner).

5. **Dataset** – 42-feature (state) + move-label pairs generated via MCTS self-play and saved to `popout_dataset.csv` for the ID3 decision tree in Phase 2.

### Observations

- The **heuristic rollout** variant generally outperforms pure random rollout because it avoids the most egregious mistakes during simulation.
- **Progressive widening** helps in positions with many legal moves (especially early game with 7 drop + 0 pop moves), concentrating resources on promising lines.
- **High exploration** ($C = 2.5$) tends to underperform with small iteration budgets because it spends too much time on suboptimal branches.
- **UCT-Tuned** shows the most consistent performance across different budgets.

### Constraints and design decisions

- The repetition-draw rule is enforced automatically in CvC mode; in human mode the player is given the option.
- The full-board draw rule is simplified: if no drop move exists and the current player has pop moves, the AI will always choose to pop (it never declares draw unless a repetition occurs).
- Game states are copied (not mutated) for MCTS, making the implementation correct but slightly slower than an in-place approach.
- The rollout depth cap (200 moves) prevents infinite loops in degenerate states.

In [ ]:
# Final demo: HvC game (uncomment to play interactively in a terminal)
# from main import main
# main()

# Or run a CvC game with the two best-performing variants
ai_best   = MCTS(iterations=600, C=math.sqrt(2), rollout='heuristic', name='Best-Heuristic')
ai_tuned  = MCTS(iterations=600, C=math.sqrt(2), tuned=True,         name='UCT-Tuned')

winner, moves = auto_play(ai_best, ai_tuned, verbose=True)
sym = PopOutGame.player_symbol(winner) if winner else 'Draw'
print(f'Final result: {sym}  in {moves} moves')